In [1]:
import os
import dotenv
dotenv.load_dotenv()
from langchain.chat_models import init_chat_model

In [2]:
api = os.getenv("GOOGLE_API_KEY")

In [3]:
model = init_chat_model(
    "google_genai:gemini-3-flash-preview",
    max_output_tokens=500,
    api_key=api
)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## Task - 1 (build a simple chatbot)

In [9]:
#chat model
response = model.invoke("list colors in rainbow")

if isinstance(response.content, list):
    print("".join(part["text"] for part in response.content if "text" in part))
else:
    print(response.content)

The traditional colors of the rainbow, in order, are:

1.  **Red**
2.  **Orange**
3.  **Yellow**
4.  **Green**
5.  **Blue**
6.  **Indigo**
7.  **Violet**

A popular way to remember this order is using the acronym **ROY G. BIV**.


## Task - 2 (integrate tools to the base model)

In [5]:
#WEATHER API TOOL
import requests
from langchain.tools import tool
import os
import dotenv

dotenv.load_dotenv()

WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")

@tool
def get_weather(location: str) -> str:
    """Get real-time weather using WeatherAPI.com"""
    url = "http://api.weatherapi.com/v1/current.json"
    params = {
        "key": WEATHER_API_KEY,
        "q": location
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        return f"Could not fetch weather for {location}"

    data = response.json()

    city = data["location"]["name"]
    country = data["location"]["country"]
    temp = data["current"]["temp_c"]
    condition = data["current"]["condition"]["text"]

    return f"The weather in {city}, {country} is {condition} with temperature {temp}°C."

location=input("Enter location for weather information: ")
result = get_weather.invoke({"location":location})
print(result)


The weather in Bangalore, India is Sunny with temperature 25.1°C.


In [6]:
#movie info with structured output tool
import json

json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

model_with_structure = model.with_structured_output(
    json_schema,
    method="json_schema",
)
response = model_with_structure.invoke("Provide details about the movie Kgf")
print(response) 

{'title': 'K.G.F: Chapter 1', 'year': 2018, 'director': 'Prashanth Neel', 'rating': 8.2}


In [7]:
#web search tool
import os
import dotenv
dotenv.load_dotenv()
from tavily import TavilyClient

api = os.getenv("TAVILY_API_KEY")


@tool("web search")
def search(query: str) -> str:
    """web search and return the top result."""
    tavily_client = TavilyClient(api_key=api)
    response = tavily_client.search(query=query, max_results=1)
    return response["results"][0]["content"] 

result = search.invoke({"query":"What is tesla stock price currently?"})
print(result)

The current Tesla(TSLA) stock price is $433.33, with a market capitalization of 1.43T. The stock trades at a price-to-earnings (P/E) ratio of 289.30.


In [ ]:
#google finance tool for real time stock price


In [ ]:
#google drive for performing rag

In [ ]:
#industry analysis tool


## Task - 3(Add helper functions)

In [8]:
#fetching date tool
from datetime import datetime
from langchain.tools import tool

@tool
def get_today_date(query:str=None) -> str:
    """Returns today's date  in d-m-y format"""
    now=datetime.now()
    return now.strftime("%d-%m-%y")

response=get_today_date.invoke({})
print(response)
  

08-01-26


In [ ]:
#formatting result  of web search tool
import os
import dotenv
import json
dotenv.load_dotenv()
from tavily import TavilyClient

api = os.getenv("TAVILY_API_KEY")


@tool("web search")
def search(query: str) -> str:
    """web search and return the top result."""
    tavily_client = TavilyClient(api_key=api)
    response = tavily_client.search(query=query, max_results=1)
    result =response.get("results",[])
    if not result:
        return "No results found."

    top = result[0] 
    
    return (
        f"Title: {top.get('title')}\n"
        f"Content: {top.get('content')}\n"
        f"Url: {top.get('url')}\n"
    ) 
    

result = search.invoke({"query":"what is tesla stock price?"})
print(result)


Title: Buy Tesla Stock - TSLA Stock Price, Quote & News | SoFi
Content: Tesla (TSLA). $431 41Market Price. -$1.55 (-0.36%)Change. $498.8352 week high. $214.2552 week low. 1D. 1W.
Url: https://www.sofi.com/invest/stock/TSLA/



In [78]:
#summarize
import os
import dotenv
dotenv.load_dotenv()
from tavily import TavilyClient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain.tools import tool
import json

llm = ChatGoogleGenerativeAI(
    model = "gemini-3-flash-preview",
    temperature = 1.0,
    max_output_tokens = 500
)

def search(query: str) -> str:
    """web search and return the top result."""
    tavily_client = TavilyClient(api_key=api)
    response = tavily_client.search(query=query, max_results=1)
    result =response.get("results",[])
    return response["results"][0]["content"] 


#runnable lambda helps in creating chains (auto summarization of results)
search_runnable = RunnableLambda(search) 

summary_prompt = PromptTemplate(
    input_variables=["content"],
    template="""
    You are a research assistant
    summarize the following information into:
    - 3 concise bullet points
    -focus only on key facts and trends
    - no extra explanation
    content:{content}
"""
)

chain = (
    search_runnable 
    | (lambda x:{"content":x})
    | summary_prompt
    | llm
)

query = "Explain about inflation crisis of venezula as of early 2026"

result = chain.invoke(query)
print(result.content)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[{'type': 'text', 'text': "* President Nicolas Maduro faces increasing US military pressure and international calls for his resignation.\n* The Venezuelan currency depreciated by 71% against the US dollar between January and September 2025.\n* The IMF projects Venezuela's annual inflation rate will reach 254% by the end of 2025.", 'extras': {'signature': 'ErcJCrQJAXLI2nw9LPOunIiDmYD1sd1v2K5s4RNr8Yv3vyUomyXrPFcX6OuASfgPYZJK3T/x6mR0ZhfZx2/y27/dFIX8XdJC1HnYqNwZx8/UJbFT6N22/XG/dBRtUwburJUmOvjMUojblOzkJl6RX7g1ImDcFAR5Ba/oc60+ia4A1T7Db3DF8G7BaO3Q4tlJTGl0blhRhmQT432DxBxcxMBMkidiovunCJkxcZCgvvtAWawDwU2W609a1P7IQRa4c4NrD/J7MO9j5jSBD4tPFjclDcf5c4clLFdN9YfSZdxEYnEW780IMbvDtoaxBFslCDRwRjhQH014nM28FCURdcU8OEcElseN5uwiRveBqFzuIjgDwGuH+72qN+MhX+MJTVNeGU/omp2Ywkni/eDm/GjBvVCmHQNs2U3JLRhw1ZCpx7ZuzzYHxOiVQlVYIjrVo5KhT1DqOB2T6Ui69pkKB4koitlvfVNt5uscahlc5IleXSijlztS1/cNmVYQq5U5c6BQDd6mGhM6wtDl8/3jT+7Kscb/D1m9wsadIfS2mzyS20/3ejpwLdGb0xVEaGbWT2T+41ECHkhCU3UxatS06Iw9qU0qzQkpEykOmXn3orVNpXzXOIvuNRmDcj6+HZ/0ji

In [ ]:
#deduplication
